In [ ]:
import pandas as pd

names = ["00.data", "01.data", "02.data", "95-01.data", "95.data", "96.data", "97.data", "98.data", "99.data"]
dfs = []

dataset = "96.data"
# for name in names:
#     df = pd.read_csv(name, sep=" ")
#     dfs.append(df)
# df  = pd.concat(dfs, ignore_index=True)
df = pd.read_csv(dataset, sep=" ")

# only use used columns
columns = [ "ortp", "no", "silica", "daph_lit", "temp", "light_m","phyto", "date"]
df = df[columns]

columns_list = list(df.columns)
df.columns

Index(['ortp', 'no', 'silica', 'daph_lit', 'temp', 'light_m', 'phyto', 'date'], dtype='str')

# plots of different interpolation techniques 

In [ ]:
from odestimate.gp.regressor import GP, heuristic_gp_fit
import matplotlib.pyplot as plt
import numpy as np


columns_list = ["phyto"]

t_obs = df["date"].to_numpy()
# rescale
t_obs = t_obs - t_obs[0]

y_obs = df[columns_list].to_numpy().T # (n_vars, n_time)

# (sub)sample values from t_obs and y_obs
n_samples = 30


mask = np.random.choice(np.arange(0, len(t_obs), 1), n_samples, replace=False)
t_gp = t_obs[mask]
y_gp = y_obs[:,mask]

# train gp on n_samples points
gp_heuristic, t_h, y_h = heuristic_gp_fit(t_obs, y_obs, max_points=n_samples, n_init=n_samples // 3, n_added=1,verbose=1, n_restarts_optimizer = 10, kernel_kwargs = {"noise_var_bounds" : (1e-5, 0.1), "length_scale_bounds" : (1, 100)})
gp_classic = GP(t_gp, y_gp, n_restarts_optimizer = 10, kernel_kwargs = {"noise_var_bounds" : (1e-5, 0.1), "length_scale_bounds" : (1, 100)})

gps = [(gp_heuristic, "heuristic"), (gp_classic, "classic")]

print(f"t: {t_gp.shape}; y: {y_gp.shape}")



heuristic_gp_fit:  83%|████████▎ | 25/30 [00:07<00:01,  3.31it/s]/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter noise_var is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
heuristic_gp_fit:  87%|████████▋ | 26/30 [00:07<00:01,  3.30it/s]/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter noise_var is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
heuristic_gp_fit:  93%|█████████▎| 28/30 [00:08<00:00,  3.03it/s]/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter noi

t: (30,); y: (1, 30)


/home/urhp/Documents/PyBM/venv/lib/python3.14/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter noise_var is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [ ]:
def mse(gp):
    return np.mean((y_obs - gp(t_obs))**2)

print(f"MSE(classic)={mse(gp_classic).item()}")
print(f"MSE(heuristic)={mse(gp_heuristic).item()}")

MSE(classic)=1.8278058115349294
MSE(heuristic)=1.9904148046819268


In [ ]:
# %matplotlib widget
#interactive plot


def index(column):
    return columns_list.index(column)

# plot 
for column in columns_list:
    i =  index(column)
    #fig, (left, right) = plt.subplots(1, 2)
    # plot data points 
    plt.scatter(t_obs, df[column], label="data", color="black",s=15)
    # plot gp datapoints
    plt.scatter(t_gp, y_gp[i], label="gp train set", color="red", s=15)
    plt.scatter(t_h, y_h[i], label="gp train set", color="green", s=15)
    # plot mean 
    mean = gp_classic(t_obs).T[i]
    plt.plot(t_obs, mean, label="clasic gp", color="blue")
    # plot var 
    # std = gp.std(t_obs).T[i]
    # plt.plot(t_obs, mean - std, label="std", color="blue", ls="--")
    # plt.plot(t_obs, mean + std, label="std", color="blue", ls="--")

    # plot mean of heuristic
    mean = gp_heuristic(t_obs).T[i]
    plt.plot(t_obs, mean, label="heuristic gp", color="green")

    plt.legend()
    plt.title(f"{column}") # - l={round(gp_classic.gps[i].gpr.kernel_.length_scale)}, σ_f²= {round(gp_classic.gps[i].gpr.kernel_.signal_var)}, σ_n²= {round(gp_classic.gps[i].gpr.kernel_.noise_var)}" )
    plt.show()
    print(f"Variable {column}:\n\tclassisc gp: l={round(gp_classic.gps[i].gpr.kernel_.length_scale)}, σ_f²= {round(gp_classic.gps[i].gpr.kernel_.signal_var)}, σ_n²= {round(gp_classic.gps[i].gpr.kernel_.noise_var)}")
    print(f"\theuristic: l={round(gp_heuristic.gps[i].gpr.kernel_.length_scale)}, σ_f²= {round(gp_heuristic.gps[i].gpr.kernel_.signal_var)}, σ_n²= {round(gp_heuristic.gps[i].gpr.kernel_.noise_var)}")


RuntimeError: 'widget' is not a recognised GUI loop or backend name